# Belief-persistence evaluation on Colab

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload `data/cam_front_images.zip` (the 804 front-camera images, 118 MB) to Google Drive at `MyDrive/cmpe249/`.

Results are written to that Drive folder, and `run` skips finished sequences, so if Colab disconnects
just reconnect and run the cells again; it continues where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/cmpe249"   # holds cam_front_images.zip; results go here too
BRANCH = "feat/eval-pipeline"

# Run once per model: Qwen first, then switch to the LLaVA lines and run from step 3 again.
MODEL, TAG = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b"
# MODEL, TAG = "llava-hf/llava-onevision-qwen2-7b-ov-hf", "llava7b"

RESULTS = f"{DRIVE_DIR}/results_{TAG}.jsonl"

## 2. Setup: Drive, code, packages, images

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!if [ -d /content/repo ]; then git -C /content/repo pull -q; else git clone -q -b {BRANCH} https://github.com/n-zad/av-vlm-belief-persistence.git /content/repo; fi
%cd /content/repo
!pip install -q -U transformers accelerate bitsandbytes
!mkdir -p data/nuscenes && unzip -q -o {DRIVE_DIR}/cam_front_images.zip -d data/nuscenes
!echo "images: $(ls data/nuscenes/samples/CAM_FRONT | wc -l) (expect 804)"

## 3. Smoke test: one sequence, all three modes

Check that the raw answers are sensible JSON before the long run. If they are empty or gibberish,
stop here and report it. If you get a CUDA out-of-memory error, add `--max-side 768` to both run commands.

In [ ]:
!rm -f /content/smoke.jsonl
!python -m belief_eval run --dataroot data/nuscenes --sequences sequences_trainval.jsonl --model {MODEL} --4bit --limit 1 --out /content/smoke.jsonl

import json
for rec in map(json.loads, open("/content/smoke.jsonl")):
    print(f"--- {rec['mode']} (evidence frame {rec['evidence_index']})")
    for f in rec["frames"]:
        print(f"t={f['t']} present={f['present']} action={f['action']} gt_action={f['gt_action']} | {f['raw'][:150]!r}")

## 4. Full run: 145 sequences × 3 modes

Expect several hours per model. Safe to re-run after a disconnect.

In [ ]:
!python -m belief_eval run --dataroot data/nuscenes --sequences sequences_trainval.jsonl --model {MODEL} --4bit --out {RESULTS}
!echo "finished records: $(wc -l < {RESULTS}) / 435"

## 5. Scores (all models run so far)

In [ ]:
!python -m belief_eval score {DRIVE_DIR}/results_*.jsonl